In [0]:
# Create struct schema for orders csv file
from pyspark.sql.types import *

order_items_schema = StructType([
    StructField("order_item_id", IntegerType(),True),
    StructField("order_id", IntegerType(), True),
    StructField("product", IntegerType(), True),
    StructField("qty", IntegerType(), True),
    StructField("price", IntegerType(), True)
])

In [0]:
# autoload csv into dataframe with schema location defined

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/schemas/bronze_order_items"
    ) \
    .schema(order_items_schema) \
    .load("/Volumes/first_data_engineering_project/landing/retail_files/order_items/")

In [0]:
# add bronze layer metadata columns to the dataFrame for ingestion and lineage tracking

from pyspark.sql import functions as F

bronze_order_items = df \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.col("_metadata.file_name")) \
    .withColumn( \
        "file_modified_time", 
        F.col("_metadata.file_modification_time"))


In [0]:
# create table with checkpoint location and add trigger

bronze_order_items.writeStream \
    .option("checkpointLocation", "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/checkpoints/bronze_order_items") \
    .trigger(availableNow=True) \
    .toTable("first_data_engineering_project.bronze.bronze_order_items")